# Attrition risk modeling

This notebook builds a baseline attrition risk model on synthetic HR data. It demonstrates model evaluation, calibration, fairness checks, and responsible interpretation. It does not recommend scoring individual employees as an HR decision tool.

The data is entirely synthetic and generated in this notebook. Replace it with your own HRIS data only after completing the risk assessment template in `03-governance/risk-assessment-template.md` and reviewing with Legal and Privacy.

## What this notebook does

1. Generates a synthetic workforce of 5,000 employees with realistic distributions.
2. Engineers features that map to real attrition signals: tenure stage, time since last promotion, comp position vs. range, manager change history, engagement trend.
3. Trains two models side by side:
   - **Logistic regression** as the interpretable baseline, preferred unless another model is clearly better, because its drivers can be explained.
   - **Gradient boosting** as the performance reference. Useful for benchmarking but harder to defend in an employment context.
4. Evaluates discrimination performance (AUC) and calibration.
5. Runs a fairness audit across demographic segments using disparate impact analysis.
6. Reports results at segment level. Individual scores are used only to validate the model.

## What this notebook does NOT do

- It does not deploy a model. Production deployment requires data infrastructure, integration with your HRIS, monitoring, and ongoing fairness audits that are out of scope for a notebook.
- It does not justify acting on individual scores. Aggregate analysis can prompt inquiry into work conditions, not employee-specific intervention.
- It does not address the prediction-to-intervention gap. Predictive association does not establish that an intervention would improve retention.

## Limitations

- **Synthetic data flatters the model.** The outcome here is generated from a relationship similar to the one the model fits, so discrimination and calibration look better than they would on real organizational data, where drivers are partly unmeasured, noisy, and changing. The results show how to evaluate a model, not how well one would perform.
- **Predicting individuals has organizational consequences.** Employees may not know they are being scored. A high score can prompt an inappropriate intervention, or become self-fulfilling if it changes how a manager treats someone. Features such as tenure, level, department, or work arrangement can act as proxies for protected characteristics. A score that reaches promotion, pay, or performance decisions turns a retention signal into an employment decision. That is why the output here is segment-level.

For the governance framework that wraps a model like this, see `03-governance/ai-use-policy.md` and `03-governance/risk-assessment-template.md`.

## Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Setup complete.')

Setup complete.


## Synthetic workforce generation

Generate 5,000 employees with realistic distributions. Attrition is modeled as a function of several factors with intentional non-linearities and interactions that resemble real HR data patterns.

In real use, this entire section is replaced by a query against your HRIS. The features below are common candidates for attrition models. Which ones predict anything in your organization is an empirical question to test, not an assumption.

In [2]:
N = 5000

df = pd.DataFrame({
    'employee_id': range(1, N + 1),
    'tenure_months': np.clip(np.random.exponential(scale=36, size=N), 1, 240).astype(int),
    'age': np.clip(np.random.normal(38, 10, N), 22, 65).astype(int),
    'level': np.random.choice([1, 2, 3, 4, 5], N, p=[0.30, 0.30, 0.20, 0.15, 0.05]),
    'department': np.random.choice(['Engineering', 'Sales', 'Marketing', 'Customer Success', 'G&A'],
                                    N, p=[0.40, 0.20, 0.10, 0.20, 0.10]),
    'gender': np.random.choice(['F', 'M', 'NB'], N, p=[0.45, 0.53, 0.02]),
    'last_engagement_score': np.clip(np.random.normal(7.2, 1.5, N), 1, 10).round(1),
    'engagement_trend': np.random.choice(['improving', 'stable', 'declining'], N, p=[0.25, 0.55, 0.20]),
    'months_since_promotion': np.clip(np.random.exponential(scale=24, size=N), 0, 120).astype(int),
    'months_since_comp_change': np.clip(np.random.exponential(scale=12, size=N), 0, 60).astype(int),
    'comp_vs_midpoint_pct': np.clip(np.random.normal(0, 12, N), -30, 30).round(1),
    'manager_changes_last_year': np.random.choice([0, 1, 2, 3], N, p=[0.65, 0.25, 0.08, 0.02]),
    'remote_status': np.random.choice(['remote', 'hybrid', 'office'], N, p=[0.35, 0.45, 0.20]),
})

df.head()

,employee_id,tenure_months,age,level,department,gender,last_engagement_score,engagement_trend,months_since_promotion,months_since_comp_change,comp_vs_midpoint_pct,manager_changes_last_year,remote_status
0,1,16,32,1,G&A,F,5.5,improving,19,4,14.9,0,hybrid
1,2,108,22,1,Customer Success,M,5.7,declining,29,9,0.8,0,remote
2,3,47,33,4,G&A,M,6.8,stable,21,28,-0.0,1,hybrid
3,4,32,47,2,Engineering,M,8.8,declining,27,21,22.8,0,hybrid
4,5,6,43,2,Engineering,M,8.8,stable,7,3,-16.3,2,hybrid


### Generating the attrition outcome

The synthetic outcome is driven by a logistic combination of features. Tenure has a non-linear effect (highest risk at 12-36 months). Engagement trend matters more than absolute score. Comp position matters less than time since last comp change (recency, not absolute).

In [3]:
def attrition_logit(row):
    score = -2.5

    if 12 <= row['tenure_months'] <= 36:
        score += 1.2
    elif row['tenure_months'] < 12:
        score += 0.4
    elif row['tenure_months'] > 120:
        score -= 0.6

    trend_effects = {'improving': -0.8, 'stable': 0.0, 'declining': 1.1}
    score += trend_effects[row['engagement_trend']]
    score += -0.15 * (row['last_engagement_score'] - 7)

    if row['months_since_promotion'] > 36:
        score += 0.6
    if row['months_since_comp_change'] > 18:
        score += 0.5

    score += 0.4 * row['manager_changes_last_year']

    if row['comp_vs_midpoint_pct'] < -10:
        score += 0.4

    dept_effects = {'Sales': 0.5, 'Customer Success': 0.2, 'Engineering': 0.0,
                    'Marketing': 0.1, 'G&A': -0.1}
    score += dept_effects[row['department']]

    return score

df['logit'] = df.apply(attrition_logit, axis=1)
df['attrition_prob'] = 1 / (1 + np.exp(-df['logit']))
df['will_attrit'] = (np.random.random(N) < df['attrition_prob']).astype(int)

print(f'Synthetic attrition rate: {df["will_attrit"].mean():.1%}')

Synthetic attrition rate: 24.0%


## Feature engineering

The raw fields above need transformation before they become useful model inputs. Most attrition signal lives in derived features, not raw values.

In [4]:
feature_df = pd.get_dummies(df, columns=['department', 'engagement_trend', 'remote_status'],
                            prefix=['dept', 'trend', 'work'], drop_first=True)

feature_df['tenure_under_12mo'] = (feature_df['tenure_months'] < 12).astype(int)
feature_df['tenure_12_to_36mo'] = ((feature_df['tenure_months'] >= 12) &
                                    (feature_df['tenure_months'] <= 36)).astype(int)
feature_df['tenure_over_10yr'] = (feature_df['tenure_months'] > 120).astype(int)

feature_df['promo_stale'] = (feature_df['months_since_promotion'] > 36).astype(int)
feature_df['comp_stale'] = (feature_df['months_since_comp_change'] > 18).astype(int)
feature_df['underpaid'] = (feature_df['comp_vs_midpoint_pct'] < -10).astype(int)

exclude_cols = ['employee_id', 'gender', 'age', 'logit', 'attrition_prob', 'will_attrit']
feature_cols = [c for c in feature_df.columns if c not in exclude_cols]

print(f'Total features: {len(feature_cols)}')
print('Note: gender and age are intentionally EXCLUDED from features. Retained only for the post-hoc fairness audit.')
print('Excluding them does not remove proxy effects, which is why the audit below still runs on them.')

Total features: 21
Note: gender and age are intentionally EXCLUDED from features. Retained only for the post-hoc fairness audit.
Excluding them does not remove proxy effects, which is why the audit below still runs on them.


## Train both models

We train a logistic regression baseline and a gradient boosting reference. Prefer the logistic model unless the other is clearly better, because its drivers can be explained to the people who use the results.

In [5]:
X = feature_df[feature_cols]
y = feature_df['will_attrit']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)

logit = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
logit.fit(X_train, y_train)
logit_proba = logit.predict_proba(X_test)[:, 1]

gbm = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_SEED)
gbm.fit(X_train, y_train)
gbm_proba = gbm.predict_proba(X_test)[:, 1]

logit_auc = roc_auc_score(y_test, logit_proba)
gbm_auc = roc_auc_score(y_test, gbm_proba)
print(f'Logistic regression AUC:  {logit_auc:.3f}')
print(f'Gradient boosting AUC:    {gbm_auc:.3f}')
print(f'Gap (GBM minus logistic): {gbm_auc - logit_auc:+.3f}')
print()
print('AUC measures how well scores rank people who left above people who stayed.')
print('It says nothing about whether any single prediction is right.')

Logistic regression AUC:  0.751
Gradient boosting AUC:    0.734
Gap (GBM minus logistic): -0.017

AUC measures how well scores rank people who left above people who stayed.
It says nothing about whether any single prediction is right.


## What drives the score

The logistic coefficients tell you what the model is paying attention to. Each flag is compared with its baseline group, shown in parentheses.

In [6]:
LABELS = {
    'tenure_months': 'Tenure (per month)',
    'level': 'Job level (per level)',
    'last_engagement_score': 'Latest engagement score (per point)',
    'months_since_promotion': 'Months since last promotion (per month)',
    'months_since_comp_change': 'Months since last pay change (per month)',
    'comp_vs_midpoint_pct': 'Pay vs. range midpoint (per percentage point)',
    'manager_changes_last_year': 'Manager changes in the past year (per change)',
    'dept_Engineering': 'In Engineering (vs. Customer Success)',
    'dept_G&A': 'In G&A (vs. Customer Success)',
    'dept_Marketing': 'In Marketing (vs. Customer Success)',
    'dept_Sales': 'In Sales (vs. Customer Success)',
    'trend_improving': 'Engagement improving (vs. declining)',
    'trend_stable': 'Engagement stable (vs. declining)',
    'work_office': 'Works in the office (vs. hybrid)',
    'work_remote': 'Works remotely (vs. hybrid)',
    'tenure_under_12mo': 'Tenure under 1 year',
    'tenure_12_to_36mo': 'Tenure 1 to 3 years',
    'tenure_over_10yr': 'Tenure over 10 years',
    'promo_stale': 'No promotion in over 3 years',
    'comp_stale': 'No pay change in over 18 months',
    'underpaid': 'Paid more than 10% below range midpoint',
}
assert set(LABELS) == set(feature_cols)

coefs = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': logit.coef_[0],
}).sort_values('coefficient', key=abs, ascending=False).head(10)

print('Top 10 drivers by absolute coefficient:')
print()
for _, row in coefs.iterrows():
    direction = 'raises' if row['coefficient'] > 0 else 'lowers'
    print(f"  {LABELS[row['feature']]:<48}  {row['coefficient']:+.3f}  ({direction} predicted risk)")

Top 10 drivers by absolute coefficient:

  Engagement improving (vs. declining)              -2.039  (lowers predicted risk)
  Tenure 1 to 3 years                               +1.268  (raises predicted risk)
  Engagement stable (vs. declining)                 -1.176  (lowers predicted risk)
  No promotion in over 3 years                      +0.646  (raises predicted risk)
  Tenure under 1 year                               +0.535  (raises predicted risk)
  No pay change in over 18 months                   +0.534  (raises predicted risk)
  Paid more than 10% below range midpoint           +0.490  (raises predicted risk)
  Manager changes in the past year (per change)     +0.372  (raises predicted risk)
  Tenure over 10 years                              -0.365  (lowers predicted risk)
  In G&A (vs. Customer Success)                     -0.337  (lowers predicted risk)


## Calibration

A model can have high AUC and still be uncalibrated. If your model says someone has a 30% attrition risk and the observed rate for similar employees is 60%, your scores are misleading even though the rank ordering is correct.

Calibration and ranking answer different validation questions. Individual predictions are checked internally against observed outcomes; they are not distributed to managers.

In [7]:
logit_frac, logit_mean = calibration_curve(y_test, logit_proba, n_bins=10, strategy='quantile')

print(f'Logistic Brier score: {brier_score_loss(y_test, logit_proba):.4f}  (lower is better)')
print(f'Gradient boosting Brier: {brier_score_loss(y_test, gbm_proba):.4f}')
print()
print('Calibration curve (predicted vs. observed) for the logistic model:')
print(f'  {"Predicted":<14}{"Observed":<14}{"Status"}')
# strict=True: calibration_curve returns paired arrays of equal length, so an
# unequal zip would mean sklearn changed its contract. Assert it rather than
# silently truncating to the shorter one.
for pred, obs in zip(logit_mean, logit_frac, strict=True):
    gap = abs(pred - obs)
    status = 'good' if gap < 0.05 else 'check' if gap < 0.10 else 'recalibrate'
    print(f'  {pred:.2f}          {obs:.2f}          {status}')

Logistic Brier score: 0.1554  (lower is better)
Gradient boosting Brier: 0.1588

Calibration curve (predicted vs. observed) for the logistic model:
  Predicted     Observed      Status
  0.05          0.02          good
  0.08          0.06          good
  0.11          0.11          good
  0.14          0.16          good
  0.18          0.20          good
  0.22          0.23          good
  0.27          0.29          good
  0.33          0.34          good
  0.43          0.41          good
  0.61          0.58          good


## Fairness audit

The most important section of this notebook. A score-producing HR model that is not audited for disparate impact is a legal and ethical liability. Audit every release. Audit every quarter in production.

We check whether the flag rate (share of scores above 0.30, an illustrative review threshold) differs meaningfully across protected groups. The four-fifths rule comes from the federal Uniform Guidelines on Employee Selection Procedures ([29 CFR 1607.4(D)](https://www.ecfr.gov/current/title-29/subtitle-B/chapter-XIV/part-1607/section-1607.4)): a group's selection rate below 80% of the highest group's rate will generally be regarded by federal agencies as evidence of adverse impact. It is a screening rule of thumb, not a legal safe harbor. The same regulation says smaller differences can still be adverse impact when they are statistically and practically significant. Applying it to an attrition flag, rather than a hiring or promotion decision, is a borrowed review threshold.

In [8]:
test_indices = X_test.index
audit_df = df.loc[test_indices].copy()
audit_df['score'] = logit_proba
audit_df['flagged_high_risk'] = (audit_df['score'] >= 0.30).astype(int)

print('Fairness audit: high-risk flag rate by gender')
print()
by_gender = audit_df.groupby('gender').agg(
    n=('employee_id', 'count'),
    flag_rate=('flagged_high_risk', 'mean'),
    actual_attrit_rate=('will_attrit', 'mean'),
).round(3)
print(by_gender.to_string())
print()

max_rate = by_gender['flag_rate'].max()
min_rate = by_gender['flag_rate'].min()
ratio = min_rate / max_rate if max_rate > 0 else 1.0

print(f'Min/max flag rate ratio: {ratio:.2%}')
print('4/5ths threshold:        80.00%')
print()
if ratio >= 0.80:
    print('PASS: 4/5ths rule satisfied.')
    print('This does NOT mean the model is fair. It means no obvious disparate impact on this segmentation.')
else:
    print('FAIL: 4/5ths rule violated. Model must not deploy until investigated.')
    if by_gender['n'].min() < 100:
        print(f"Caution: the smallest group has n={int(by_gender['n'].min())}. A ratio built on a group that small")
        print('swings widely between random splits. Confirm with more data before drawing a conclusion,')
        print('but do not wave it away: in production a FAIL on a small group is still a reason to investigate.')
print()

# Age was excluded from the features but is still audited: dropping a protected
# characteristic from the inputs does not show the outputs are neutral.
audit_df['age_band'] = pd.cut(audit_df['age'], bins=[0, 29, 39, 49, 100], labels=['under 30', '30-39', '40-49', '50+'])
by_age = audit_df.groupby('age_band', observed=True).agg(
    n=('employee_id', 'count'),
    flag_rate=('flagged_high_risk', 'mean'),
    actual_attrit_rate=('will_attrit', 'mean'),
).round(3)
print('Same check by age band')
print()
print(by_age.to_string())
print()
age_ratio = by_age['flag_rate'].min() / by_age['flag_rate'].max() if by_age['flag_rate'].max() > 0 else 1.0
age_verdict = 'PASS' if age_ratio >= 0.80 else 'FAIL'
print(f'Min/max flag rate ratio: {age_ratio:.2%}  ({age_verdict} against the 80% threshold)')
print()
print('Also run this analysis across: department, tenure bucket, race/ethnicity if collected,')
print('and any other protected characteristic relevant to your jurisdiction.')

Fairness audit: high-risk flag rate by gender

          n  flag_rate  actual_attrit_rate
gender                                    
F       510      0.292               0.249
M       710      0.287               0.232
NB       30      0.367               0.267

Min/max flag rate ratio: 78.20%
4/5ths threshold:        80.00%

FAIL: 4/5ths rule violated. Model must not deploy until investigated.
Caution: the smallest group has n=30. A ratio built on a group that small
swings widely between random splits. Confirm with more data before drawing a conclusion,
but do not wave it away: in production a FAIL on a small group is still a reason to investigate.

Same check by age band

            n  flag_rate  actual_attrit_rate
age_band                                    
under 30  281      0.295               0.217
30-39     464      0.291               0.239
40-49     367      0.281               0.248
50+       138      0.312               0.268

Min/max flag rate ratio: 90.06%  (PASS against

### Beyond selection rates: does the score mean the same thing for everyone?

The four-fifths check above compares how often each group gets flagged. That catches one failure mode and misses two others that matter more for a risk score:

- **Calibration by group.** A score of 0.30 should mean roughly a 30 percent observed attrition rate in every group. If the model is over-confident for one group, that group is over-flagged or under-flagged.
- **Error-rate parity.** Equal flag rates can hide unequal mistakes. A group with a higher false negative rate is the group whose flight risk the model systematically misses.

Neither replaces the four-fifths rule; they answer different questions, and a deployment review should look at all three. The thresholds below are review triggers, not legal standards.

Watch group sizes. A group with a few dozen people in the test set will show error-rate swings that are noise, not signal; the cell flags that and says what to do about it.

In [9]:
from sklearn.metrics import confusion_matrix

def group_report(frame, group_col, score_col='score', flag_col='flagged_high_risk', outcome_col='will_attrit'):
    rows = []
    for name, g in frame.groupby(group_col):
        tn, fp, fn, tp = confusion_matrix(g[outcome_col], g[flag_col], labels=[0, 1]).ravel()
        rows.append({
            group_col: name,
            'n': len(g),
            'brier': brier_score_loss(g[outcome_col], g[score_col]),
            'mean_score': g[score_col].mean(),
            'observed_rate': g[outcome_col].mean(),
            'false_negative_rate': fn / (fn + tp) if (fn + tp) else float('nan'),
            'false_positive_rate': fp / (fp + tn) if (fp + tn) else float('nan'),
        })
    return pd.DataFrame(rows).set_index(group_col).round(3)

report = group_report(audit_df, 'gender')
print('Calibration and error rates by gender (test set)')
print()
print(report.to_string())
print()

calib_gap = (report['mean_score'] - report['observed_rate']).abs().max()
fnr_gap = report['false_negative_rate'].max() - report['false_negative_rate'].min()
fpr_gap = report['false_positive_rate'].max() - report['false_positive_rate'].min()

print(f'Largest mean-score vs observed-rate gap in any group: {calib_gap:.3f}  (review if > 0.05)')
print(f'False negative rate spread across groups:              {fnr_gap:.3f}  (review if > 0.05)')
print(f'False positive rate spread across groups:              {fpr_gap:.3f}  (review if > 0.05)')
print()
for label, gap in (('calibration', calib_gap), ('false negative rate', fnr_gap), ('false positive rate', fpr_gap)):
    print(f"{'REVIEW' if gap > 0.05 else 'OK    '}  {label}")
print()
small = report[report['n'] < 100]
if len(small):
    groups = ', '.join(map(str, small.index))
    print(f'Small-group caution: {groups} has fewer than 100 people in the test set,')
    print('so its error rates are unstable and can swing by 0.1 or more between random')
    print('splits. Pool across periods or use confidence intervals before treating a gap')
    print('driven by a small group as a finding.')
    print()
print('Repeat this by department, tenure bucket, age band, and any protected characteristic you lawfully hold.')
print('A REVIEW result is a reason to investigate the feature set and threshold before deployment, not a verdict.')

Calibration and error rates by gender (test set)

          n  brier  mean_score  observed_rate  false_negative_rate  false_positive_rate
gender                                                                                 
F       510  0.163       0.238          0.249                0.488                0.219
M       710  0.150       0.243          0.232                0.436                0.204
NB       30  0.161       0.255          0.267                0.250                0.227

Largest mean-score vs observed-rate gap in any group: 0.012  (review if > 0.05)
False negative rate spread across groups:              0.238  (review if > 0.05)
False positive rate spread across groups:              0.023  (review if > 0.05)

OK      calibration
REVIEW  false negative rate
OK      false positive rate

Small-group caution: NB has fewer than 100 people in the test set,
so its error rates are unstable and can swing by 0.1 or more between random
splits. Pool across periods or use confidence 

### Calibration within groups, not just on average

The table above compares each group's mean predicted score to its observed rate. That is
calibration-in-the-large, and it is the weak form of the check. Two groups can both have a mean
score that matches their observed rate exactly while being badly miscalibrated inside that
average: over-predicting at the low end and under-predicting at the high end cancels out in the
mean and disappears from the table.

That matters because anything built on the scores, including the segment-level reporting below,
inherits miscalibration at whatever part of the score range a segment sits. The
markdown above says a 0.30 score should mean a 30 percent observed rate; this cell is the version
of that check which can actually catch a violation.

Expected calibration error (ECE) bins each group's predictions and averages the gap between
predicted and observed within each bin, weighted by bin size. A low mean gap with a high ECE is
exactly the case the table above hides.


In [10]:
def expected_calibration_error(y_true, y_prob, n_bins=5):
    """Binned |predicted - observed|, weighted by bin population.

    Equal-width bins over [0, 1] rather than quantile bins, so the bins mean the
    same thing across groups and the numbers are comparable between them. Bins
    with no members contribute nothing. Returns (ece, per_bin_rows).
    """
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    total = len(y_true)
    ece, rows = 0.0, []
    for lo, hi in zip(edges[:-1], edges[1:], strict=True):
        # Include the right edge only in the final bin, so a prediction of
        # exactly 1.0 is counted once rather than dropped.
        in_bin = (y_prob >= lo) & ((y_prob < hi) | (hi == 1.0) & (y_prob <= hi))
        n = int(in_bin.sum())
        if n == 0:
            rows.append({'bin': f'[{lo:.1f},{hi:.1f})', 'n': 0, 'predicted': np.nan,
                         'observed': np.nan, 'gap': np.nan})
            continue
        predicted = y_prob[in_bin].mean()
        observed = y_true[in_bin].mean()
        gap = abs(predicted - observed)
        ece += (n / total) * gap
        rows.append({'bin': f'[{lo:.1f},{hi:.1f})', 'n': n, 'predicted': round(predicted, 3),
                     'observed': round(observed, 3), 'gap': round(gap, 3)})
    return ece, rows


print('Expected calibration error by group (equal-width bins, test set)')
print()
ece_by_group = {}
for name, g in audit_df.groupby('gender'):
    ece, rows = expected_calibration_error(g['will_attrit'], g['score'])
    ece_by_group[name] = ece
    mean_gap = abs(g['score'].mean() - g['will_attrit'].mean())
    print(f'{name}  (n={len(g)})')
    print(f"    mean-score gap: {mean_gap:.3f}   ECE: {ece:.3f}")
    print('    ' + pd.DataFrame(rows).to_string(index=False).replace('\n', '\n    '))
    print()

ece_spread = max(ece_by_group.values()) - min(ece_by_group.values())
worst_group, worst_ece = max(ece_by_group.items(), key=lambda kv: kv[1])
print(f'Worst within-group ECE: {worst_ece:.3f} ({worst_group})   (review if > 0.05)')
print(f'ECE spread across groups: {ece_spread:.3f}          (review if > 0.05)')
print()
print('Read this next to the mean-score gap above. A small mean gap with a large ECE means the')
print('group average is fine and the individual scores are not, which is the case that matters')
print('because managers act on individual scores. Watch the per-bin n column: a bin with a')
print('handful of people produces a gap that is noise, and ECE weights by bin size precisely so')
print('those bins do not dominate the headline number.')


Expected calibration error by group (equal-width bins, test set)

F  (n=510)
    mean-score gap: 0.011   ECE: 0.018
          bin   n  predicted  observed   gap
    [0.0,0.2) 263      0.116     0.122 0.006
    [0.2,0.4) 161      0.285     0.323 0.038
    [0.4,0.6)  68      0.478     0.471 0.007
    [0.6,0.8)  17      0.675     0.647 0.028
    [0.8,1.0)   1      0.827     0.000 0.827

M  (n=710)
    mean-score gap: 0.010   ECE: 0.021
          bin   n  predicted  observed   gap
    [0.0,0.2) 367      0.112     0.109 0.003
    [0.2,0.4) 217      0.286     0.286 0.000
    [0.4,0.6)  84      0.488     0.369 0.119
    [0.6,0.8)  40      0.668     0.750 0.082
    [0.8,1.0)   2      0.819     1.000 0.181

NB  (n=30)
    mean-score gap: 0.011   ECE: 0.065
          bin  n  predicted  observed   gap
    [0.0,0.2) 11      0.112     0.091 0.022
    [0.2,0.4) 15      0.304     0.267 0.037
    [0.4,0.6)  4      0.466     0.750 0.284
    [0.6,0.8)  0        NaN       NaN   NaN
    [0.8,1.0)  0      

## What reaches people: segment-level results

The model scores individuals only so that calibration and error rates can be checked, as above. Those individual scores are a validation step, not an HR decision interface. What goes to HR leaders is aggregate: which segments show elevated predicted attrition, how that compares with what was observed, and what drives it, in plain language. Segments below a minimum size are suppressed.

In [11]:
MIN_GROUP = 20  # illustrative; set the real minimum with Privacy

audit_df['tenure_band'] = pd.cut(
    audit_df['tenure_months'], bins=[0, 12, 37, 121, 10_000],
    labels=['Under 1 year', '1 to 3 years', '3 to 10 years', 'Over 10 years'], right=False,
)
segments = (
    audit_df.groupby(['department', 'tenure_band'], observed=True)
    .agg(people=('score', 'size'), mean_predicted=('score', 'mean'), observed_rate=('will_attrit', 'mean'))
    .reset_index()
)
shown = segments[segments['people'] >= MIN_GROUP].sort_values('mean_predicted', ascending=False)

suppressed = len(segments) - len(shown)
print(f'Segments with at least {MIN_GROUP} people (test set); {suppressed} smaller segments suppressed.')
print()
print(shown.round(3).to_string(index=False))
print()
print('What raises or lowers predicted attrition across the workforce:')
for _, row in coefs.head(6).iterrows():
    direction = 'raises' if row['coefficient'] > 0 else 'lowers'
    print(f"  - {LABELS[row['feature']]}: {direction} predicted risk")

Segments with at least 20 people (test set); 4 smaller segments suppressed.

      department   tenure_band  people  mean_predicted  observed_rate
           Sales  1 to 3 years      83           0.423          0.482
Customer Success  1 to 3 years      89           0.376          0.382
     Engineering  1 to 3 years     163           0.331          0.319
       Marketing  1 to 3 years      51           0.320          0.373
           Sales  Under 1 year      86           0.288          0.314
             G&A  1 to 3 years      36           0.263          0.250
Customer Success  Under 1 year      71           0.245          0.211
             G&A  Under 1 year      34           0.234          0.235
     Engineering  Under 1 year     151           0.195          0.139
       Marketing  Under 1 year      37           0.180          0.108
           Sales 3 to 10 years      64           0.178          0.203
Customer Success 3 to 10 years      84           0.161          0.119
             

## Governance non-negotiables before deploying

Before any aggregate findings from a model like this reach business users:

1. **Approval gate.** Run through the [risk assessment template](../03-governance/risk-assessment-template.md). This use case is medium-to-high risk and requires HR Leadership, Legal, and Privacy sign-off.

2. **Fairness audit cadence.** Quarterly at minimum. Whenever data drifts. Whenever the model is retrained. Document each audit. Failing audits halt the model.

3. **Score interpretation discipline.** Only if calibration holds in the evaluated population should predictions near 60% correspond to roughly 60% observed departures. Check this internally; do not interpret an individual prediction as a reason for intervention.

4. **No action on individual scores.** No emails, no flags to managers, no compensation or performance decisions. Individual scores stay inside model validation; what reaches HR is segment-level.

5. **Employee transparency.** Employees should know that an attrition risk model exists, what data it uses, and that they can request human review of any consequential decision involving model output. Where the model output feeds a decision with legal or similarly significant effect on someone in the EU, GDPR Article 22 limits solely automated decisions and requires meaningful human involvement. Transparency and human review are also increasingly required by state law. Confirm the exact obligations with Legal for each jurisdiction.

6. **Score retention policy.** Scores are time-sensitive. Stale scores influence decisions in ways that are hard to audit. Default to deleting scores older than 90 days unless there is a specific retention reason.

7. **No manager access to individual scores.** A score a manager can see risks becoming performance feedback, which it is not.

## Adapting this for your organization

1. **Establish purpose and data permissions first.** Use organizational data only after approval of the question, access, minimization, and retention. Field availability does not justify its use.

2. **Recalibrate the feature engineering.** The synthetic relationships are illustrative, not defaults for a real workforce. Test feature assumptions and proxy effects on appropriately governed data.

3. **Set your own review threshold.** The 0.30 threshold here is illustrative and is used only to compute flag rates for the fairness audit.

4. **Define the workforce inquiry first.** Decide which aggregate comparisons could inform an organizational question and whether a model adds value over observed trends. Validate internally before sharing suppressed segment reports; do not assume retention benefits.

5. **Plan for failure modes.** What happens if the model is biased on a segment you have not measured? What happens if miscalibrated segment estimates misdirect an organizational inquiry? Define how to withdraw misleading reports and correct their interpretation.